# 05: Production Monitoring & ML ROI

## Learning Objectives
- Build monitoring dashboards for ML systems
- Calculate business ROI of deployed models
- Implement alerting for model degradation
- Create stakeholder-ready presentations

## Roadmap
Production Metrics → Alerting → Streamlit Dashboard → ROI Calculation → Executive Summary

In [ ]:
import os, json, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib.gridspec as gridspec
import seaborn as sns
from datetime import datetime, timedelta
from pathlib import Path
warnings.filterwarnings('ignore')

import sys; sys.path.insert(0, '../../..')
from shared.viz_utils import setup_style
setup_style()
print("Ready")

---
## Section 1: What to Monitor in Production

### The 4 Levels of ML Monitoring

| Level | What to track | Example metrics |
|-------|--------------|-----------------|
| **Infrastructure** | System health | CPU, memory, latency, error rate |
| **Model inputs** | Data quality | Null rate, out-of-range values, drift |
| **Model outputs** | Predictions | Score distribution, prediction rate |
| **Business** | Impact | Churn rate, revenue, conversion |

### The Monitoring Stack
```
Application Logs → CloudWatch → Alarm → SNS → Slack/Email
Model Metrics   → MLflow    → Alert → Retrain Trigger
Business KPIs   → Dashboard → Report → Stakeholders
```

### When to Alert
- P99 latency > 500ms for 5+ minutes → page on-call
- Model AUC drops > 5pp vs baseline → auto-create retrain ticket
- PSI > 0.2 on any feature → send Slack alert
- Error rate > 1% → page on-call immediately

In [ ]:
def generate_production_metrics(days: int = 90, seed: int = 42) -> pd.DataFrame:
    """Simulate 90 days of model production metrics."""
    np.random.seed(seed)
    dates = [datetime(2024, 7, 1) + timedelta(days=i) for i in range(days)]
    
    # Model performance degradation after day 60 (simulated concept drift)
    base_auc = 0.82
    auc = np.array([
        base_auc + np.random.normal(0, 0.008) if i < 60 
        else base_auc - (i-60)*0.002 + np.random.normal(0, 0.008)
        for i in range(days)
    ]).clip(0.65, 0.92)
    
    # Traffic patterns (weekday/weekend)
    daily_requests = np.array([
        1200 + 300 * np.random.randn() if d.weekday() < 5
        else 600 + 150 * np.random.randn()
        for d in dates
    ]).clip(100).astype(int)
    
    # Latency (occasional spikes)
    p50_latency = np.random.normal(18, 3, days).clip(8)
    p95_latency = p50_latency + np.random.exponential(20, days)
    spike_days = np.random.choice(days, 5, replace=False)
    p95_latency[spike_days] *= 4  # Latency spikes
    
    # Error rate
    error_rate = np.random.beta(1, 200, days)  # ~0.5% most days
    incident_days = np.random.choice(days, 2, replace=False)
    error_rate[incident_days] = np.random.uniform(0.05, 0.15)  # 2 incidents
    
    # Feature drift (PSI)
    psi_tenure = np.random.exponential(0.05, days) * (1 + np.arange(days)/60)
    psi_monthly = np.random.exponential(0.04, days) * (1 + np.arange(days)/90)
    
    return pd.DataFrame({
        'date': dates,
        'daily_requests': daily_requests,
        'model_auc': auc.round(4),
        'p50_latency_ms': p50_latency.round(1),
        'p95_latency_ms': p95_latency.round(1),
        'error_rate': error_rate.round(5),
        'psi_tenure': psi_tenure.round(4),
        'psi_monthly': psi_monthly.round(4),
        'churn_rate_actual': (0.25 + np.arange(days)*0.0003 + np.random.normal(0, 0.01, days)).clip(0.15, 0.40).round(4),
        'model_predicted_churn': (0.23 + np.arange(days)*0.0001 + np.random.normal(0, 0.01, days)).clip(0.15, 0.40).round(4),
    })

metrics = generate_production_metrics()
print(f"Generated {len(metrics)} days of metrics")
print(metrics.describe())

In [ ]:
fig = plt.figure(figsize=(16, 12))
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.4, wspace=0.35)

# 1. Model AUC over time
ax1 = fig.add_subplot(gs[0, :2])
ax1.plot(metrics['date'], metrics['model_auc'], color='#3498db', linewidth=1.5)
ax1.axhline(0.82, color='gray', linestyle='--', alpha=0.7, label='Baseline AUC')
ax1.axhline(0.78, color='red', linestyle='--', alpha=0.7, label='Alert threshold')
ax1.axvspan(metrics['date'].iloc[60], metrics['date'].iloc[-1], alpha=0.05, color='red')
ax1.text(metrics['date'].iloc[62], 0.83, 'Drift detected', fontsize=9, color='red')
ax1.set_title('Model AUC (90 days)', fontweight='bold')
ax1.set_ylabel('AUC'); ax1.legend(fontsize=9); ax1.set_ylim(0.65, 0.92)

# 2. Daily requests
ax2 = fig.add_subplot(gs[0, 2])
ax2.bar(range(len(metrics)), metrics['daily_requests'], color='#2ecc71', alpha=0.7, width=1)
ax2.set_title('Daily Requests', fontweight='bold')
ax2.set_ylabel('Requests')
ax2.set_xticks([0, 30, 60, 89])
ax2.set_xticklabels(['Jul 1', 'Jul 31', 'Aug 30', 'Sep 28'])

# 3. P95 Latency
ax3 = fig.add_subplot(gs[1, :2])
ax3.fill_between(metrics['date'], metrics['p50_latency_ms'], metrics['p95_latency_ms'],
                  alpha=0.3, color='#e67e22', label='P50-P95 range')
ax3.plot(metrics['date'], metrics['p95_latency_ms'], color='#e67e22', linewidth=1, label='P95')
ax3.axhline(200, color='red', linestyle='--', alpha=0.7, label='Alert: 200ms')
ax3.set_title('API Latency (ms)', fontweight='bold')
ax3.set_ylabel('Latency (ms)'); ax3.legend(fontsize=9)

# 4. Error rate
ax4 = fig.add_subplot(gs[1, 2])
ax4.fill_between(range(len(metrics)), metrics['error_rate']*100, alpha=0.7, color='#e74c3c')
ax4.set_title('Error Rate (%)', fontweight='bold')
ax4.set_ylabel('Error Rate (%)')
ax4.set_xticks([0, 30, 60, 89])
ax4.set_xticklabels(['Jul', 'Aug', 'Sep', 'Oct'], fontsize=8)

# 5. Feature drift (PSI)
ax5 = fig.add_subplot(gs[2, :2])
ax5.plot(metrics['date'], metrics['psi_tenure'], label='tenure PSI', color='#9b59b6', linewidth=1.5)
ax5.plot(metrics['date'], metrics['psi_monthly'], label='monthly_charges PSI', color='#1abc9c', linewidth=1.5)
ax5.axhline(0.1, color='orange', linestyle='--', alpha=0.7, label='Monitor threshold')
ax5.axhline(0.2, color='red', linestyle='--', alpha=0.7, label='Retrain threshold')
ax5.set_title('Feature Drift (PSI)', fontweight='bold')
ax5.set_ylabel('PSI'); ax5.legend(fontsize=9)

# 6. Prediction vs actual churn
ax6 = fig.add_subplot(gs[2, 2])
ax6.plot(range(len(metrics)), metrics['churn_rate_actual']*100, label='Actual', color='#e74c3c', linewidth=1.5)
ax6.plot(range(len(metrics)), metrics['model_predicted_churn']*100, label='Predicted', 
          color='#3498db', linewidth=1.5, linestyle='--')
ax6.set_title('Churn Rate: Actual vs Predicted', fontweight='bold')
ax6.set_ylabel('Churn Rate (%)'); ax6.legend(fontsize=9)
ax6.set_xticks([0, 30, 60, 89])
ax6.set_xticklabels(['Jul', 'Aug', 'Sep', 'Oct'], fontsize=8)

fig.suptitle('ML Production Dashboard — Churn Model v1.0', fontsize=14, fontweight='bold', y=1.01)
plt.show()

---
## Section 2: Automated Alerting

### Alert Rules (implemented as functions)
In production: these run in CloudWatch, Grafana, or a monitoring service like Datadog.

In [ ]:
class AlertSystem:
    """Simple alert system for model monitoring."""
    
    def __init__(self, rules: dict):
        self.rules = rules
        self.alerts_fired = []
    
    def check(self, metrics_row: dict) -> list:
        """Check metrics against rules and return fired alerts."""
        fired = []
        for metric, (threshold, comparison, severity) in self.rules.items():
            value = metrics_row.get(metric)
            if value is None:
                continue
            triggered = (
                (comparison == '>' and value > threshold) or
                (comparison == '<' and value < threshold)
            )
            if triggered:
                alert = {
                    'metric': metric,
                    'value': round(value, 4),
                    'threshold': threshold,
                    'severity': severity,
                    'message': f"[{severity}] {metric} = {value:.4f} ({comparison} {threshold})"
                }
                fired.append(alert)
                self.alerts_fired.append(alert)
        return fired

# Configure alert rules
alert_system = AlertSystem(rules={
    'model_auc': (0.78, '<', 'CRITICAL'),     # Retrain if AUC drops below 0.78
    'p95_latency_ms': (200, '>', 'WARNING'),   # Alert if P95 > 200ms
    'error_rate': (0.02, '>', 'CRITICAL'),      # Page on-call if > 2% errors
    'psi_tenure': (0.2, '>', 'WARNING'),        # Drift alert
    'psi_monthly': (0.2, '>', 'WARNING'),       # Drift alert
})

# Check last 7 days
print("Alert Check — Last 7 Days")
print("="*60)
for _, row in metrics.tail(7).iterrows():
    alerts = alert_system.check(row.to_dict())
    date_str = row['date'].strftime('%Y-%m-%d')
    if alerts:
        for a in alerts:
            print(f"  {date_str}: {a['message']}")
    else:
        print(f"  {date_str}: All clear (AUC={row['model_auc']:.4f})")

print(f"\nTotal alerts fired in 90 days: {len(alert_system.alerts_fired)}")

---
## Section 3: Calculating ML ROI

### The Business Case Formula
```
ROI = (Revenue Generated - Total Cost) / Total Cost × 100%

Revenue Generated = Customers Retained × Monthly Revenue × Model Attribution
Total Cost = Development Cost + Infrastructure Cost + Maintenance Cost
```

### Attribution Problem
Not all retained customers would have churned without intervention.
We need to estimate **incremental retention** — the customers saved BECAUSE of the model.
This is why A/B tests are so valuable: they directly measure incremental impact.

In [ ]:
def calculate_ml_roi(
    # Model performance
    total_customers: int = 50_000,
    model_precision: float = 0.68,       # Of customers flagged, 68% would really churn
    model_recall: float = 0.72,          # Of real churners, model catches 72%
    
    # Business parameters
    monthly_churn_rate: float = 0.025,   # 2.5% churn/month without intervention
    avg_monthly_revenue: float = 65.0,   # $/customer/month
    retention_offer_cost: float = 25.0,  # Cost of retention offer
    offer_acceptance_rate: float = 0.35, # 35% of flagged customers accept offer
    offer_effectiveness: float = 0.60,   # 60% of acceptors stay (vs would-have-churned)
    
    # Model costs
    dev_cost_months: float = 3,           # 3 engineer-months at $15K/month
    engineer_monthly_salary: float = 15_000,
    monthly_infra_cost: float = 500,      # SageMaker + S3 + monitoring
    
    months: int = 12,
) -> dict:
    """Calculate annualized ML ROI."""
    
    # Expected churners per month (without model)
    expected_churners = total_customers * monthly_churn_rate
    
    # Customers the model flags for intervention
    # Model catches recall% of real churners, but also flags some false positives
    true_positives = expected_churners * model_recall
    flagged_customers = true_positives / model_precision  # = TP / precision
    false_positives = flagged_customers - true_positives
    
    # Retention outcomes
    accepted_offers = flagged_customers * offer_acceptance_rate
    customers_retained = accepted_offers * offer_effectiveness  # incremental retention
    
    # Revenue impact
    revenue_saved_per_month = customers_retained * avg_monthly_revenue
    offer_cost_per_month = flagged_customers * offer_acceptance_rate * retention_offer_cost
    net_revenue_per_month = revenue_saved_per_month - offer_cost_per_month
    net_revenue_annual = net_revenue_per_month * months
    
    # Costs
    dev_cost = dev_cost_months * engineer_monthly_salary
    infra_cost_annual = monthly_infra_cost * months
    total_cost = dev_cost + infra_cost_annual
    
    roi = (net_revenue_annual - total_cost) / total_cost * 100
    payback_months = total_cost / max(net_revenue_per_month, 1)
    
    return {
        # Business metrics
        'monthly_expected_churners': round(expected_churners),
        'customers_flagged_per_month': round(flagged_customers),
        'true_positives_per_month': round(true_positives),
        'false_positives_per_month': round(false_positives),
        'customers_retained_per_month': round(customers_retained, 1),
        
        # Financial
        'monthly_revenue_saved': round(revenue_saved_per_month, 0),
        'monthly_offer_cost': round(offer_cost_per_month, 0),
        'monthly_net_revenue': round(net_revenue_per_month, 0),
        'annual_net_revenue': round(net_revenue_annual, 0),
        
        # Costs
        'development_cost': round(dev_cost, 0),
        'annual_infra_cost': round(infra_cost_annual, 0),
        'total_cost': round(total_cost, 0),
        
        # ROI
        'roi_pct': round(roi, 1),
        'payback_months': round(payback_months, 1),
        'npv_estimate': round(net_revenue_annual - total_cost, 0),
    }

roi = calculate_ml_roi()
print("ML ROI Analysis — Churn Prediction Model")
print("="*50)
print(f"\nModel Impact per Month:")
print(f"  Expected churners:          {roi['monthly_expected_churners']:,}")
print(f"  Customers flagged:          {roi['customers_flagged_per_month']:,}")
print(f"  Customers retained:         {roi['customers_retained_per_month']:,}")
print(f"\nFinancials (Monthly):")
print(f"  Revenue saved:            ${roi['monthly_revenue_saved']:,.0f}")
print(f"  Retention offer cost:     ${roi['monthly_offer_cost']:,.0f}")
print(f"  Net revenue:              ${roi['monthly_net_revenue']:,.0f}")
print(f"\nCosts (One-time + Annual):")
print(f"  Development:              ${roi['development_cost']:,.0f}")
print(f"  Annual infrastructure:    ${roi['annual_infra_cost']:,.0f}")
print(f"  Total cost:               ${roi['total_cost']:,.0f}")
print(f"\nReturn on Investment:")
print(f"  Annual net revenue:       ${roi['annual_net_revenue']:,.0f}")
print(f"  ROI:                      {roi['roi_pct']:.1f}%")
print(f"  Payback period:           {roi['payback_months']:.1f} months")
print(f"  Estimated NPV:            ${roi['npv_estimate']:,.0f}")

In [ ]:
# Sensitivity: how does ROI change with different assumptions?
sensitivities = []

for precision in [0.55, 0.65, 0.75, 0.85]:
    for recall in [0.55, 0.65, 0.75, 0.85]:
        r = calculate_ml_roi(model_precision=precision, model_recall=recall)
        sensitivities.append({
            'precision': precision,
            'recall': recall,
            'roi_pct': r['roi_pct'],
            'monthly_net': r['monthly_net_revenue'],
        })

sens_df = pd.DataFrame(sensitivities)
pivot = sens_df.pivot(index='recall', columns='precision', values='roi_pct')

plt.figure(figsize=(9, 6))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='RdYlGn', center=200,
            xticklabels=[f'{p:.0%}' for p in pivot.columns],
            yticklabels=[f'{r:.0%}' for r in pivot.index])
plt.title('ROI (%) vs Model Precision & Recall\n(Sensitivity Analysis)', fontsize=13)
plt.xlabel('Precision'); plt.ylabel('Recall')
plt.tight_layout()
plt.show()
print("\nKey insight: Precision matters more than recall for ROI when offer costs are non-trivial.")

In [ ]:
streamlit_code = '''"""
Production Monitoring Dashboard — run with: streamlit run dashboard.py
"""

import streamlit as st
import pandas as pd, numpy as np, plotly.express as px, plotly.graph_objects as go
from datetime import datetime, timedelta

st.set_page_config(page_title="ML Monitor", layout="wide")
st.title("Churn Model — Production Dashboard")

# Sidebar
with st.sidebar:
    st.header("Controls")
    days = st.slider("Days to show", 7, 90, 30)
    alert_threshold_auc = st.slider("AUC Alert Threshold", 0.70, 0.85, 0.78)

# Load data (replace with real monitoring DB)
@st.cache_data(ttl=300)
def load_metrics():
    # In production: query CloudWatch, MLflow, or your metrics DB
    np.random.seed(42)
    dates = [datetime.now() - timedelta(days=i) for i in range(90)][::-1]
    return pd.DataFrame({
        "date": dates,
        "auc": np.clip(0.82 + np.cumsum(np.random.normal(-0.001, 0.008, 90)), 0.65, 0.92),
        "requests": np.random.poisson(1000, 90),
        "p95_ms": np.random.exponential(50, 90).clip(15, 500),
        "error_rate": np.random.beta(1, 200, 90),
    })

df = load_metrics().tail(days)

# Metrics row
col1, col2, col3, col4 = st.columns(4)
col1.metric("Current AUC", f"{df[\'auc\'].iloc[-1]:.4f}", 
            delta=f"{df[\'auc\'].iloc[-1] - df[\'auc\'].iloc[0]:.4f}")
col2.metric("Avg Daily Requests", f"{df[\'requests\'].mean():,.0f}")
col3.metric("P95 Latency", f"{df[\'p95_ms\'].mean():.0f}ms")
col4.metric("Error Rate", f"{df[\'error_rate\'].mean()*100:.2f}%")

# AUC chart
fig = px.line(df, x="date", y="auc", title="Model AUC Over Time")
fig.add_hline(y=alert_threshold_auc, line_dash="dash", line_color="red",
              annotation_text=f"Alert threshold ({alert_threshold_auc})")
st.plotly_chart(fig, use_container_width=True)

st.success("Run this with: streamlit run dashboard.py")
'''

print("Streamlit dashboard code:")
print("Run with: pip install streamlit plotly && streamlit run dashboard.py")

---
## Key Takeaways

1. **Monitor 4 levels**: infra, inputs, outputs, business — not just model performance
2. **Alert early**: set thresholds before going to prod, not after the incident
3. **ROI framing**: stakeholders care about $, not AUC — translate metrics to dollars
4. **Sensitivity analysis**: test your ROI assumptions — which variable matters most?
5. **Dashboards**: Streamlit for internal, Grafana/CloudWatch for production
6. **Track AUC weekly**: if AUC drops > 3pp from baseline, start the retrain conversation

---
## Exercises

### Exercise 1: Alert Report
Build a function that takes the metrics DataFrame and returns a markdown-formatted daily alert report (email-ready). Include: date, all metrics, any alerts fired, and recommended actions.

### Exercise 2: ROI Sensitivity
Identify which input to `calculate_ml_roi()` has the biggest impact on final ROI. Run 100 Monte Carlo samples varying all inputs simultaneously. What's the 5th/95th percentile ROI?

### Exercise 3: Streamlit Dashboard
Install Streamlit and run the dashboard code above. Add a "Drift" tab that shows PSI for each feature with color-coded status indicators.